In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
from helper import load_env
load_env()

import os
import yaml
from crewai import Agent, Task, Crew

In [ ]:
os.environ['OPENAI_MODEL_NAME'] = 'gpt-4o-mini'

In [ ]:
files = {
    'lead_agents': 'config/lead_qualification_agents.yaml',
    'lead_tasks': 'config/lead_qualification_tasks.yaml',
    'email_agents': 'config/email_engagement_agents.yaml',
    'email_tasks': 'config/email_engagement_tasks.yaml'
}

configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

lead_agents_config = configs['lead_agents']
lead_tasks_config = configs['lead_tasks']
email_agents_config = configs['email_agents']
email_tasks_config = configs['email_tasks']

## Pydantic Models for Structured Output

In [4]:
from pydantic import BaseModel, Field
from typing import Dict, Optional, List, Set, Tuple

class LeadPersonalInfo(BaseModel):
    name: str = Field(..., description="The full name of the lead.")
    job_title: str = Field(..., description="The job title of the lead.")
    role_relevance: int = Field(..., ge=0, le=10, description="A score representing how relevant the lead's role is to the decision-making process (0-10).")
    professional_background: Optional[str] = Field(..., description="A brief description of the lead's professional background.")

class CompanyInfo(BaseModel):
    company_name: str = Field(..., description="The name of the company the lead works for.")
    industry: str = Field(..., description="The industry in which the company operates.")
    company_size: int = Field(..., description="The size of the company in terms of employee count.")
    revenue: Optional[float] = Field(None, description="The annual revenue of the company, if available.")
    market_presence: int = Field(..., ge=0, le=10, description="A score representing the company's market presence (0-10).")

class LeadScore(BaseModel):
    score: int = Field(..., ge=0, le=100, description="The final score assigned to the lead (0-100).")
    scoring_criteria: List[str] = Field(..., description="The criteria used to determine the lead's score.")
    validation_notes: Optional[str] = Field(None, description="Any notes regarding the validation of the lead score.")

class LeadScoringResult(BaseModel):
    personal_info: LeadPersonalInfo = Field(..., description="Personal information about the lead.")
    company_info: CompanyInfo = Field(..., description="Information about the lead's company.")
    lead_score: LeadScore = Field(..., description="The calculated score and related information for the lead.")

In [5]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

## Lead Qualification Crew, Agents and Tasks

In [6]:
# Creating Agents
lead_data_agent = Agent(
  config=lead_agents_config['lead_data_agent'],
  tools=[SerperDevTool(), ScrapeWebsiteTool()]
)

cultural_fit_agent = Agent(
  config=lead_agents_config['cultural_fit_agent'],
  tools=[SerperDevTool(), ScrapeWebsiteTool()]
)

scoring_validation_agent = Agent(
  config=lead_agents_config['scoring_validation_agent'],
  tools=[SerperDevTool(), ScrapeWebsiteTool()]
)

# Creating Tasks
lead_data_task = Task(
  config=lead_tasks_config['lead_data_collection'],
  agent=lead_data_agent
)

cultural_fit_task = Task(
  config=lead_tasks_config['cultural_fit_analysis'],
  agent=cultural_fit_agent
)

scoring_validation_task = Task(
  config=lead_tasks_config['lead_scoring_and_validation'],
  agent=scoring_validation_agent,
  context=[lead_data_task, cultural_fit_task],
  output_pydantic=LeadScoringResult
)

# Creating Crew
lead_scoring_crew = Crew(
  agents=[
    lead_data_agent,
    cultural_fit_agent,
    scoring_validation_agent
  ],
  tasks=[
    lead_data_task,
    cultural_fit_task,
    scoring_validation_task
  ],
  verbose=True
)

## Email Engagement Crew

In [7]:
# Creating Agents
email_content_specialist = Agent(
  config=email_agents_config['email_content_specialist']
)

engagement_strategist = Agent(
  config=email_agents_config['engagement_strategist']
)

# Creating Tasks
email_drafting = Task(
  config=email_tasks_config['email_drafting'],
  agent=email_content_specialist
)

engagement_optimization = Task(
  config=email_tasks_config['engagement_optimization'],
  agent=engagement_strategist
)

# Creating Crew
email_writing_crew = Crew(
  agents=[
    email_content_specialist,
    engagement_strategist
  ],
  tasks=[
    email_drafting,
    engagement_optimization
  ],
  verbose=True
)

2026-08-09 15:53:29,969 - 139692013337472 - __init__.py-__init__:538 - WARNING: Overriding of current TracerProvider is not allowed


## Creating Complete Sales Flow

In [8]:
from crewai import Flow
from crewai.flow.flow import listen, start

class SalesPipeline(Flow):
    @start()
    def fetch_leads(self):
        # Pull our leads from the database
        leads = [
            {
                "lead_data": {
                    "name": "João Moura",
                    "job_title": "Director of Engineering",
                    "company": "Clearbit",
                    "email": "joao@clearbit.com",
                    "use_case": "Using AI Agent to do better data enrichment."
                },
            },
        ]
        return leads

    @listen(fetch_leads)
    def score_leads(self, leads):
        scores = lead_scoring_crew.kickoff_for_each(leads)
        self.state["score_crews_results"] = scores
        return scores

    @listen(score_leads)
    def store_leads_score(self, scores):
        # Here we would store the scores in the database
        return scores

    @listen(score_leads)
    def filter_leads(self, scores):
        return [score for score in scores if score['lead_score'].score > 70]

    @listen(filter_leads)
    def write_email(self, leads):
        scored_leads = [lead.to_dict() for lead in leads]
        emails = email_writing_crew.kickoff_for_each(scored_leads)
        return emails

    @listen(write_email)
    def send_email(self, emails):
        # Here we would send the emails to the leads
        return emails

flow = SalesPipeline()

## Plotting the Flow

In [9]:
flow.plot()

Plot saved as crewai_flow.html


In [10]:
from IPython.display import IFrame

IFrame(src='./crewai_flow.html', width='150%', height=600)

## Flow Kickoff

In [17]:
emails = await flow.kickoff_async()

2026-08-09 15:56:05,314 - 139692013337472 - __init__.py-__init__:538 - WARNING: Overriding of current TracerProvider is not allowed


# Agent: Lead Data Specialist
## Task: Collect and analyze the following information about the lead:
- Personal Information:
  - Name: Obtain the full name of the lead.
  - Job Title: Determine the lead's current job title.
  - Role Relevance: Assess how relevant the lead's role is to the decision-making process on a scale from 0 to 10.
  - Professional Background: Optionally, gather a brief description of the lead's professional background.

- Company Information:
  - Company Name: Identify the name of the company the lead works for.
  - Industry: Determine the industry in which the company operates.
  - Company Size: Estimate the size of the company in terms of employee count.
  - Revenue: If available, collect information on the annual revenue of the company.
  - Market Presence: Evaluate the company's market presence on a scale from 0 to 10.

- Our Company and Product:
  - Company Name: CrewAI
  - Product: Multi-Agent Orchestration Platform
  - ICP: Enterprise companies looking int



# Agent: Cultural Fit Analyst
## Thought: I need to gather information about Clearbit's cultural values, mission, and strategic objectives to assess the alignment with CrewAI. This will involve looking for publicly available details regarding their internal culture and values, as well as any statements about their mission and goals.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"Clearbit company culture values mission statement\"}"
## Tool Output: 

Search results: Title: Leverage 100+ business data attributes
Link: https://clearbit.com/attributes
Snippet: Enrich your CRM and database, manage leads efficiently, and enable marketing personalization with over 100 business data points.
---
Title: What is Brief History of Clearbit Company?
Link: https://businessmodelcanvastemplate.com/blogs/brief-history/clearbit-brief-history?srsltid=AfmBOooCYYXu42GE0k6OtYnK99T9Uo7ANP3BjvKSRyo7Ehqwu9AibjAI
Snippet: Clearbit's name signaled its mission-clarifying the 'bits' of d



# Agent: Cultural Fit Analyst
## Thought: Thought: I need to extract more specific information about Clearbit's core values and mission to assess their culture and alignment with CrewAI. Since I don't have access to internal documents, I will search for other resources that discuss these aspects of Clearbit's culture.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"Clearbit company culture values employee engagement mission statement\"}"
## Tool Output: 

Search results: Title: 35+ Employee Engagement Mission Statement Examples
Link: https://www.performyard.com/articles/employee-engagement-mission-statement-examples
Snippet: "Engaging our employees through meaningful work, clear goals, and a shared sense of purpose to drive company success."
---
Title: How Company Culture Impacts Employee Engagement
Link: https://www.loebleadership.com/insights/how-company-culture-impacts-employee-engagement
Snippet: A strong company culture positively impacts employee engage



# Agent: Cultural Fit Analyst
## Thought: Thought: I need to gather further insights into Clearbit by analyzing employee feedback and experiences to develop a well-rounded view of their company culture and values.
## Using tool: Search the internet
## Tool Input: 
"{\"search_query\": \"Clearbit company culture employee feedback Glassdoor\"}"
## Tool Output: 

Search results: Title: Clearbit "leadership" Reviews - Glassdoor
Link: https://www.glassdoor.com/Reviews/Clearbit-leadership-Reviews-EI_IE2208101.0,8_KH9,19.htm
Snippet: 74% of Clearbit employees would recommend working there to a friend based on Glassdoor reviews. Employees also rated Clearbit 4.2 out of 5 for work life balance ...
---
Title: Clearbit Reviews (60): Pros & Cons of Working At Clearbit ... - Glassdoor
Link: https://www.glassdoor.com/Reviews/Clearbit-Reviews-E2208101.htm
Snippet: 74% of Clearbit employees would recommend working there to a friend based on Glassdoor reviews. Employees also rated Clearbit 4.2 out of 



# Agent: Cultural Fit Analyst
## Final Answer: 
```json
{
    "Cultural Fit Score": 8,
    "Supporting Analysis": {
        "Cultural Values": "Clearbit promotes innovation and teamwork, allowing employees the autonomy to excel, which resonates with CrewAI’s collaborative approach.",
        "Strategic Alignment": "Clearbit's focus on enriching data harmonizes with CrewAI's automation insights, making them well-aligned partners.",
        "Comments": "While largely compatible, ongoing efforts in diversity and employee engagement improvement can enhance cultural synergy."
    }
}
```


# Agent: Lead Scorer and Validator
## Task: Aggregate the collected data and perform the following steps: - Score Calculation: Based on predefined criteria, calculate a final lead score (0-100). Consider factors such as:
  - Role Relevance
  - Company Size
  - Market Presence
  - Cultural Fit
- Scoring Criteria Documentation: List the criteria used to determine the score. - Validation: Review the collec

2026-08-09 15:56:51,009 - 139692013337472 - __init__.py-__init__:538 - WARNING: Overriding of current TracerProvider is not allowed


# Agent: Email Content Writer
## Task: Craft a highly personalized email using the lead's name, job title, company information, and any relevant personal or company achievements. The email should speak directly to the lead's interests and the needs of their company. This is not as cold outreach as it is a follow up to a lead form, so keep it short and to the point. Don't use any salutations or closing remarks, nor too complex sentences.
Our Company and Product: - Company Name: CrewAI - Product: Multi-Agent Orchestration Platform - ICP: Enterprise companies looking into Agentic automation. - Pitch: We are a platform that allows you to orchestrate AI Agents for automations to any vertical.
Use the following information: Personal Info: {'name': 'João Moura', 'job_title': 'Director of Engineering', 'role_relevance': 8, 'professional_background': None} Company Info: {'company_name': 'Clearbit', 'industry': 'Technology', 'company_size': 200, 'revenue': 17000000.0, 'market_presence': 10} Lead

## Usage Metrics and Costs

In [18]:
import pandas as pd

# Convert UsageMetrics instance to a DataFrame
df_usage_metrics = pd.DataFrame([flow.state["score_crews_results"][0].token_usage.dict()])

# Calculate total costs
costs = 0.150 * df_usage_metrics['total_tokens'].sum() / 1_000_000
print(f"Total costs: ${costs:.4f}")

# Display the DataFrame
df_usage_metrics

Total costs: $0.0166


,total_tokens,prompt_tokens,completion_tokens,successful_requests
0,110830,103991,6839,33


In [19]:
import pandas as pd

# Convert UsageMetrics instance to a DataFrame
df_usage_metrics = pd.DataFrame([emails[0].token_usage.dict()])

# Calculate total costs
costs = 0.150 * df_usage_metrics['total_tokens'].sum() / 1_000_000
print(f"Total costs: ${costs:.4f}")

# Display the DataFrame
df_usage_metrics

Total costs: $0.0003


,total_tokens,prompt_tokens,completion_tokens,successful_requests
0,1810,1427,383,3


In [20]:
scores = flow.state["score_crews_results"]

In [21]:
import pandas as pd
from IPython.display import display, HTML

lead_scoring_result = scores[0].pydantic

# Create a dictionary with the nested structure flattened
data = {
    'Name': lead_scoring_result.personal_info.name,
    'Job Title': lead_scoring_result.personal_info.job_title,
    'Role Relevance': lead_scoring_result.personal_info.role_relevance,
    'Professional Background': lead_scoring_result.personal_info.professional_background,
    'Company Name': lead_scoring_result.company_info.company_name,
    'Industry': lead_scoring_result.company_info.industry,
    'Company Size': lead_scoring_result.company_info.company_size,
    'Revenue': lead_scoring_result.company_info.revenue,
    'Market Presence': lead_scoring_result.company_info.market_presence,
    'Lead Score': lead_scoring_result.lead_score.score,
    'Scoring Criteria': ', '.join(lead_scoring_result.lead_score.scoring_criteria),
    'Validation Notes': lead_scoring_result.lead_score.validation_notes
}

# Convert the dictionary to a DataFrame
df = pd.DataFrame.from_dict(data, orient='index', columns=['Value'])

# Reset the index to turn the original column names into a regular column
df = df.reset_index()

# Rename the index column to 'Attribute'
df = df.rename(columns={'index': 'Attribute'})

# Create HTML table with bold attributes and left-aligned values
html_table = df.style.set_properties(**{'text-align': 'left'}) \
                     .format({'Attribute': lambda x: f'<b>{x}</b>'}) \
                     .hide(axis='index') \
                     .to_html()

# Display the styled HTML table
display(HTML(html_table))

Attribute,Value
Name,João Moura
Job Title,Director of Engineering
Role Relevance,8
Professional Background,None
Company Name,Clearbit
Industry,Technology
Company Size,200
Revenue,17000000.000000
Market Presence,10
Lead Score,78


## Results

In [22]:
import textwrap

result_text = emails[0].raw
wrapped_text = textwrap.fill(result_text, width=80)
print(wrapped_text)

Unlock the potential of AI-driven automation with CrewAI. Our Multi-Agent
Orchestration Platform is crafted for enterprise companies like Clearbit,
enabling seamless orchestration of AI Agents to enhance workflow efficiency.
Ready to boost your operational capabilities?   👉 **Schedule a quick 15-minute
call with us [here](#)**. Discover how to turn your engineering initiatives into
high-impact automation.    Curious about real results?   ✨ **Check out our
success stories [here](#)** to see how similar companies achieved measurable
outcomes.  Don't miss the chance to elevate your team’s productivity. **Act
now!**


## Increasing Complexity

In [24]:
from crewai import Flow
from crewai.flow.flow import listen, start, and_, router


class SalesPipeline(Flow):

    @start()
    def fetch_leads(self):
        # Pull leads from the database
        # This is a mock; in a real-world scenario,
        # this is where you would fetch leads from a database.

        leads = [
            {
                "lead_data": {
                    "name": "João Moura",
                    "job_title": "Director of Engineering",
                    "company": "Clearbit",
                    "email": "joao@clearbit.com",
                    "use_case": "Using AI Agent to do better data enrichment.",
                },
            },
        ]

        return leads

    @listen(fetch_leads)
    def score_leads(self, leads):
        scores = lead_scoring_crew.kickoff_for_each(leads)

        self.state["score_crews_results"] = scores

        return scores

    @listen(score_leads)
    def store_leads_score(self, scores):
        # Here we would store the scores in the database
        return scores

    @listen(score_leads)
    def filter_leads(self, scores):
        return [
            score
            for score in scores
            if score["lead_score"].score > 70
        ]

    @listen(and_(filter_leads, store_leads_score))
    def log_leads(self, leads):
        print(f"Leads: {leads}")
        return leads

    @router(filter_leads)
    def count_leads(self, scores):
        if len(scores) > 10:
            return "high"
        elif len(scores) > 5:
            return "medium"
        else:
            return "low"

    @listen("high")
    def store_in_salesforce(self, leads):
        return leads

    @listen("medium")
    def send_to_sales_team(self, leads):
        return leads

    @listen("low")
    def write_email(self, leads):
        scored_leads = []

        for lead in leads:
            if hasattr(lead, "model_dump"):
                scored_leads.append(lead.model_dump())
            elif hasattr(lead, "to_dict"):
                scored_leads.append(lead.to_dict())
            else:
                scored_leads.append(lead)

        emails = email_writing_crew.kickoff_for_each(scored_leads)

        return emails

    @listen(write_email)
    def send_email(self, emails):
        # Here we would send the emails to the leads
        return emails

## Plotting the Flow

In [25]:
flow = SalesPipeline()
flow.plot()

Plot saved as crewai_flow.html


In [26]:
from IPython.display import IFrame

IFrame(src='./crewai_flow_complex.html', width='150%', height=600)